# Genetic Algorithm for Job Scheduling (R3, Reusable for R4/R5)

This notebook is built from scratch for **R3** and uses a generic GA pipeline that can be reused directly for **R4** and **R5**.

## Plan
1. Build generic data and scheduling functions.
2. Implement GA operators: selection, interval crossover, swap mutation.
3. Train for 100 generations.
4. Run R3 with fixed data.
5. Reuse the same functions for R4 and R5 random scenarios.
6. Compare best chromosome against a random pre-crossover selected chromosome.

In [ ]:
import random
from dataclasses import dataclass
from typing import List, Sequence, Tuple, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Global GA parameters (assignment defaults)
POPULATION_SIZE = 120
SELECTED_POOL_SIZE = 10
MUTATION_PROB = 0.05
GENERATIONS = 100

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


@dataclass
class GAConfig:
    population_size: int = POPULATION_SIZE
    selected_pool_size: int = SELECTED_POOL_SIZE
    mutation_prob: float = MUTATION_PROB
    generations: int = GENERATIONS
    seed: int = SEED

## Data Model

- `job_times`: 2D list/array where each row is one job and each column is one operation duration.
- `chromosome`: permutation of job indices `[0..J-1]`.
- `num_machines`: optional explicit machine count. If omitted, it defaults to number of operations.

This design remains generic so the same functions work for R3, R4, and R5.

In [ ]:
def reset_seeds(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


def to_numpy_jobs(job_times: Sequence[Sequence[int]]) -> np.ndarray:
    arr = np.asarray(job_times, dtype=int)
    if arr.ndim != 2:
        raise ValueError("job_times must be a 2D structure with shape (jobs, operations)")
    if np.any(arr < 0):
        raise ValueError("operation durations must be non-negative")
    return arr


def validate_chromosome(chromosome: Sequence[int], num_jobs: int) -> bool:
    return len(chromosome) == num_jobs and sorted(chromosome) == list(range(num_jobs))

## Generic Makespan and Fitness (Any J and M)

We use dynamic scheduling with permutation constraints:
- Jobs follow the order in the chromosome.
- Each job processes operations in sequence.
- Operation start time is `max(previous operation end for same job, machine availability)`.

If `num_machines` is not given, it defaults to number of operations. If operations exceed machines, operations wrap around machines cyclically (`op_idx % num_machines`) so the function stays generic.

In [ ]:
def compute_makespan(
    chromosome: Sequence[int],
    job_times: Sequence[Sequence[int]],
    num_machines: int | None = None,
    return_trace: bool = False,
) -> int | Tuple[int, List[Dict[str, int]]]:
    jobs = to_numpy_jobs(job_times)
    num_jobs, num_ops = jobs.shape

    if not validate_chromosome(chromosome, num_jobs):
        raise ValueError("invalid chromosome: it must be a permutation of job indices")

    machines = num_machines if num_machines is not None else num_ops
    if machines <= 0:
        raise ValueError("num_machines must be >= 1")

    machine_ready = [0] * machines
    trace: List[Dict[str, int]] = []

    for job_id in chromosome:
        job_ready = 0
        for op_idx in range(num_ops):
            m_idx = op_idx % machines
            duration = int(jobs[job_id, op_idx])
            start = max(job_ready, machine_ready[m_idx])
            end = start + duration
            machine_ready[m_idx] = end
            job_ready = end
            if return_trace:
                trace.append(
                    {
                        "job": int(job_id),
                        "op": int(op_idx),
                        "machine": int(m_idx),
                        "start": int(start),
                        "end": int(end),
                        "duration": int(duration),
                    }
                )

    makespan = int(max(machine_ready) if machine_ready else 0)
    if return_trace:
        return makespan, trace
    return makespan


def fitness_from_makespan(makespan: int) -> float:
    # Higher fitness means lower makespan.
    return 1.0 / (1.0 + makespan)


def evaluate_population(
    population: Sequence[Sequence[int]],
    job_times: Sequence[Sequence[int]],
    num_machines: int | None = None,
) -> List[Dict[str, Any]]:
    evaluated = []
    for chrom in population:
        mk = compute_makespan(chrom, job_times, num_machines)
        evaluated.append(
            {
                "chromosome": list(chrom),
                "makespan": mk,
                "fitness": fitness_from_makespan(mk),
            }
        )
    return evaluated

## Initial Population (120 Chromosomes)

We generate random permutations of job IDs and optionally deduplicate to reduce repeats.

In [ ]:
def create_initial_population(
    num_jobs: int,
    population_size: int,
    deduplicate: bool = True,
) -> List[List[int]]:
    population: List[List[int]] = []
    seen = set()

    max_unique = 1
    for i in range(2, num_jobs + 1):
        max_unique *= i

    target = min(population_size, max_unique) if deduplicate else population_size

    while len(population) < target:
        chrom = list(np.random.permutation(num_jobs))
        key = tuple(chrom)
        if deduplicate and key in seen:
            continue
        if deduplicate:
            seen.add(key)
        population.append(chrom)

    while len(population) < population_size:
        population.append(list(np.random.permutation(num_jobs)))

    return population

## Selection of M=10 Chromosomes

Minimization target is makespan, so probabilities use inverse makespan fitness.

In [ ]:
def select_pool(
    evaluated_population: Sequence[Dict[str, Any]],
    pool_size: int,
) -> List[List[int]]:
    fitness = np.array([item["fitness"] for item in evaluated_population], dtype=float)
    probs = fitness / fitness.sum()
    indices = np.random.choice(len(evaluated_population), size=pool_size, replace=False, p=probs)
    return [list(evaluated_population[i]["chromosome"]) for i in indices]

## Interval-Based Crossover (Two Cut Points)

This follows the assignment rule: keep middle segment from Parent A, remove those genes from Parent B, then fill holes in order.